In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X=data.data
y=data.target 


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Train Shape: (455, 30)
Test Shape: (114, 30)


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

param_gird = {
    "model__C": [0.01, 0.1, 1, 10],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_gird,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

Best parameters: {'model__C': 0.1}
Best CV accuracy: 0.9802197802197803


# Accuracy

In [4]:
from sklearn.metrics import accuracy_score

# Best model from GridSearchCV
best_model = grid.best_estimator_

# Predictions
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")


Train accuracy: 0.9868
Test accuracy:  0.9737


# Confusion Matrix

In [5]:
from sklearn.metrics import confusion_matrix, classification_report

cm= confusion_matrix(y_test, y_test_pred)
print("Confusion matrix:\n",cm)

# Detailed metrics
print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

Confusion matrix:
 [[40  2]
 [ 1 71]]

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.95      0.96        42
           1       0.97      0.99      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



## Confusion Matrix – Concise Insight

### Class mapping
- **0 → Malignant (critical)**
- **1 → Benign**

### Summary of outcomes
- **Malignant correctly detected:** 40  
- **Malignant missed (FN):** 2  
- **Benign wrongly flagged (FP):** 1  
- **Benign correctly detected:** 71  

### Key takeaways
- **Low false negatives (2)** → Rarely misses cancer cases (most important clinically).
- **Low false positives (1)** → Minimal unnecessary alarms.
- **High recall for malignant (0.95)** → Strong cancer detection capability.
- **High precision for malignant (0.98)** → Predictions of cancer are highly reliable.
- **Balanced metrics** → No class bias; stable generalization.

### Bottom line
The model is **accurate, well-balanced, and clinically safe**, making it suitable for **probability-based evaluation and threshold tuning using ROC–AUC**.


In [6]:
from sklearn.metrics import precision_score, f1_score, recall_score

precision = precision_score(y_test, y_test_pred)
print("Precision:",precision)
recall = recall_score(y_test, y_test_pred)
print("Recall:",recall)
f1 = f1_score(y_test, y_test_pred)
print("f1:",f1)

Precision: 0.9726027397260274
Recall: 0.9861111111111112
f1: 0.9793103448275862


# ROC Curve (Receiver Operating Characteristic)

## What is it?
The **ROC curve** is a graph that illustrates the performance of a classification model at **all classification thresholds**.
It plots two parameters:
1.  **True Positive Rate (TPR)**: Same as **Recall** or **Sensitivity**.
    $$\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
2.  **False Positive Rate (FPR)**: The proportion of actual negatives that were incorrectly classified as positive.
    $$\text{FPR} = \frac{\text{FP}}{\text{TN} + \text{FP}}$$

## Why do we use it?
- **Threshold Independence**: Accuracy depends on a fixed threshold (usually 0.5). The ROC curve shows how the model performs **regardless of the threshold chosen**.
- **Trade-off Visualization**: It visualizes the trade-off between catching positive cases (High TPR) vs. raising false alarms (High FPR).
- **Imbalanced Data**: It is often more informative than accuracy when classes are imbalanced.

## When to use it?
- When you care about **ranking** predictions (probabilities) rather than just hard classes.
- When the costs of False Positives and False Negatives are different, and you need to choose an optimal threshold.

## What is AUC (Area Under the Curve)?
- **AUC** provides an aggregate measure of performance across all possible classification thresholds.
- **Range**: 0.0 to 1.0.
- **Interpretation**:
  - **AUC = 1.0**: Perfect model. Can perfectly separate positive and negative classes.
  - **AUC = 0.5**: Random guessing. No discriminative power.
  - **AUC < 0.5**: Worse than random (model is flipping classes).
  - **AUC > 0.7**: Generally considered acceptable.
  - **AUC > 0.9**: Excellent model.

In [7]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# 1. Get predicted probabilities for the positive class (1)
# best_model is the GridSearchCV estimator from the previous cells
y_probs = best_model.predict_proba(X_test)[:, 1]

# 2. Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

# 3. Calculate AUC (Area Under Curve)
auc_score = roc_auc_score(y_test, y_probs)

# 4. Plot the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"ROC-AUC Score: {auc_score:.4f}")